# 🔬 AI Skin: Dataset Exploration, Preprocessing & Medical Augmentation
### Final-Year Academic Project: Intelligent Skin Diseases Detection
**Backbone:** MobileNetV2 | **Dataset Standard:** HAM10000 / ISIC (7 Diagnostic Classes)

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image

# Add project root to sys.path
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ConfigManager
from src.data.split import DatasetSplitter
from src.data.imbalance import ImbalanceAnalyzer
from src.data.augmentation import get_training_transforms, denormalize_tensor
from src.data.dataloader import create_dataloaders

cfg = ConfigManager()
print(f"AI Skin Framework Loaded! Model: {cfg.model_config.architecture} | Classes: {cfg.dataset_config.num_classes}")

In [ ]:
# 1. Dataset Scan & Class Imbalance Analysis
data_dir = PROJECT_ROOT / 'data' / 'sample_dataset'
class_mapping = {c.code: idx for idx, c in enumerate(cfg.dataset_config.classes)}
splitter = DatasetSplitter(validate_files=True)
df, mapping = splitter.discover_dataset(data_dir, class_mapping=class_mapping)

class_names = {idx: c.name for idx, c in enumerate(cfg.dataset_config.classes)}
analyzer = ImbalanceAnalyzer()
analysis = analyzer.analyze_distribution(df['class_idx'].tolist(), class_names=class_names)
analyzer.print_summary(analysis)

In [ ]:
# 2. Clinically Bounded Dermatoscopic Augmentation Showcase
sample_img_path = PROJECT_ROOT / 'data' / 'sample_images' / 'sample_lesion.jpg'
pil_img = Image.open(sample_img_path).convert('RGB')

train_transform = get_training_transforms(image_size=(224, 224))

fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
axes[0].imshow(pil_img.resize((224, 224)))
axes[0].set_title('Original Lesion\n(224x224)', fontweight='bold', color='#0f766e')
axes[0].axis('off')

for i in range(1, 6):
    tensor = train_transform(pil_img)
    denorm = denormalize_tensor(tensor)
    axes[i].imshow(denorm.permute(1, 2, 0).numpy())
    axes[i].set_title(f'Augmented #{i}')
    axes[i].axis('off')

plt.suptitle('Medical-Grade Dermatoscopic Augmentations (Preserving Pathology)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()